# PDF Audiobook — optional Kokoro GPU run

This notebook keeps the desktop app unchanged. It clones the project, runs the existing headless pipeline with one Kokoro narrator on the selected Colab GPU, and downloads the verified M4B. Free Colab runtimes can disconnect or have no GPU; checkpoints live under `/content` unless you change the workspace path.

**Previewing a voice.** Pick a voice in the panel, edit the sample text (e.g. paste a passage from your book), and click **Preview voice**, or select voices under **Compare with** and click **Compare voices** to hear them one after another; no PDF needed.

**Getting started.** Choose **Runtime > Run all**. When setup finishes, the panel lets you preview voices straight away; nothing needs to be uploaded first. Choose your book in the panel, then run the last cell to generate.

**Your own voices.** In the panel's **Add a voice** section, choose a short clip of any voice (MP3, WAV, M4A, a video file, ...) and click **Add to library**. Each clip is saved to `MyDrive/AudiobookFree/voices` on your Google Drive, so your library is still there next session. Custom voices show up at the top of the voice list marked with ★; preview, compare and generate with them like any other voice. They are narrated by Chatterbox at speed 1.0; pick its **Custom voice model** in the panel: Turbo (default; understands tags such as `[laugh]` and `[sigh]` in the text), Original (largest and slowest; use its **Expression** slider instead of tags), or Nano (fastest, lowest quality). Only use voices you own or have permission to use, and keep audiobooks made with someone else's voice to yourself.

**Character voices (optional).** Set **Narration** at the top of the panel to **Character voices (Gemini)**. Gemini reads your book and works out who says each quote; the book's words are never changed, and everything outside quotes is read by the narrator. It uses Colab's built-in Gemini through your Google login, so no key is needed; if that is unavailable or hits its limit, paste a free API key from [aistudio.google.com](https://aistudio.google.com/apikey). Click **Make script**: Gemini also sums up who each character is, notes how they should sound, and recommends a voice for each. Click **Preview cast** to hear every character read one of their own lines, change any voice you like (pick from any built-in or ★ custom voice, or **Narrator voice**), then run the last cell. Minor characters with fewer lines than **Own voice from** (5 by default) are read by the narrator. The script is saved to `MyDrive/AudiobookFree/cast` as a text file you can edit to fix a line, and every rendered line is kept there too, so a rerun after a disconnect continues where it stopped.

**Designing voices.** Under **Design a voice**, describe any voice (or pick a preset such as *Gruff old man* or *Charming woman*) and click **Design voice**. Qwen3-TTS VoiceDesign invents the voice and speaks a short sample, which is saved to your library; Chatterbox then clones that sample to narrate, so the voice stays the same for the whole book. With **Character voices**, each main character gets a voice designed from Gemini's description of them by default. The designer installs and downloads its model (about 4 GB) the first time you use it in a session.

**Changing settings on a run that already started.** A conversion is bound to the voice, speed, PDF and chapter settings it began with. Leave the panel's **Start new** checkbox unticked to *resume* that conversion after a disconnect — rerun the generate cell with the same settings and it picks up from the last completed chunk. To change the voice, speed, PDF or chapter settings you must tick **Start new**, which discards that conversion's checkpoints (its entire active conversion tree) and begins again from the first chapter. A run started with an older version of this notebook cannot be resumed; tick **Start new** once.

In [ ]:
import subprocess, torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. Choose Runtime > Change runtime type > T4 GPU, then rerun.')
print(torch.cuda.get_device_name(0))

In [ ]:
# Linux tools used by the existing PDF parser and M4B finalizer.
!apt-get -qq update
!apt-get -qq install -y ffmpeg espeak-ng
!pip -q install uv

In [ ]:
# Colab's runtime Python may be 3.12; provision the project's pinned Python 3.11 environment.
REPO = '/content/AudiobookFree'
BRANCH = 'main'
!rm -rf /content/AudiobookFree
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/AugustZhang1/AudiobookFree.git', REPO], check=True)
!uv python install 3.11
!uv venv --allow-existing --python 3.11 /content/audiobook-venv
PY = '/content/audiobook-venv/bin/python'
# PyPI's newest torch targets a newer CUDA than Colab's GPU driver supports, so the venv would see no GPU.
# Install the same torch build Colab ships (it matches the runtime's driver) before anything can pull another one.
TORCH_VERSION = torch.__version__.split('+')[0]
TORCH_INDEX = 'https://download.pytorch.org/whl/cu' + torch.version.cuda.replace('.', '')
!uv pip install --python {PY} "torch=={TORCH_VERSION}" --index-url {TORCH_INDEX}
# Every other package version is pinned in colab/constraints.txt, so a new upstream release cannot break setup.
!uv pip install --python {PY} -e {REPO} -c {REPO}/colab/constraints.txt
!uv pip install --python {PY} "torch=={TORCH_VERSION}" -r {REPO}/colab/requirements.in -c {REPO}/colab/constraints.txt
gpu_check = subprocess.run([PY, '-c', 'import torch, kokoro; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())'], capture_output=True, text=True)
print(gpu_check.stdout + gpu_check.stderr, end='')
if gpu_check.returncode != 0 or not gpu_check.stdout.strip().endswith('True'):
    raise RuntimeError('Setup failed: Kokoro did not install or the environment cannot see the GPU; see the output above.')

In [ ]:
# Voice cloning for your own custom voices (Chatterbox). If this fails, the built-in voices still work.
# Chatterbox pins different torch/transformers versions than Kokoro, so it gets its own environment (a few GB; takes a few minutes).
import subprocess
REPO = '/content/AudiobookFree'
CB_PY = REPO + '/engine_envs/chatterbox/.venv/bin/python'
!uv sync --frozen --project {REPO}/engine_envs/chatterbox --python 3.11
!uv pip install --python {CB_PY} -e {REPO}
cb_check = subprocess.run([CB_PY, '-c', 'import torch, chatterbox.tts_turbo, pdf_audiobook.colab; print(torch.cuda.is_available())'], capture_output=True, text=True)
print(cb_check.stdout + cb_check.stderr, end='')
if cb_check.returncode != 0 or not cb_check.stdout.strip().endswith('True'):
    print('Voice cloning is unavailable (see the output above); the built-in voices still work.')
else:
    print('Voice cloning is ready.')

## Choose a voice and settings

Use the panel below to choose your narrator voice, adjust speed, audition sample text, add your own voices, and choose your book.
Changes apply immediately — the generate cell reads the panel's current values when it runs, so no rerun is needed.
The first preview downloads the Kokoro model (or the Chatterbox model for a ★ custom voice), so it is slower than subsequent previews.
This cell connects your Google Drive to load your voice library; allow access when asked.
If the runtime restarts, rerun this cell to restore the panel.

In [ ]:
import glob
import hashlib
import html
import importlib
import io
import json
import logging
import os
import re
import shutil
import subprocess
import sys
import time
import wave
from IPython.display import Audio, HTML, display
import ipywidgets as widgets

VOICE_IDS = [
    "af_heart", "af_alloy", "af_aoede", "af_bella", "af_jessica", "af_kore", "af_nicole",
    "af_nova", "af_river", "af_sarah", "af_sky", "am_adam", "am_echo", "am_eric",
    "am_fenrir", "am_liam", "am_michael", "am_onyx", "am_puck", "am_santa",
    "bf_alice", "bf_emma", "bf_isabella", "bf_lily", "bm_daniel", "bm_fable",
    "bm_george", "bm_lewis",
]

_ACCENT_MAP = {'a': 'US', 'b': 'UK'}
_GENDER_MAP = {'f': 'female', 'm': 'male'}


def _format_voice_label(voice_id: str) -> str:
    prefix, name = voice_id.split('_', 1)
    accent = _ACCENT_MAP.get(prefix[0], prefix[0])
    gender = _GENDER_MAP.get(prefix[1], prefix[1])
    return f"{name.title()} — {accent} {gender}"


VOICE_OPTIONS = [(_format_voice_label(vid), vid) for vid in VOICE_IDS]
VOICE_LABELS = {vid: label for label, vid in VOICE_OPTIONS}

PY = '/content/audiobook-venv/bin/python'  # set by setup; repeated so this cell works after a runtime restart
CB_PY = '/content/AudiobookFree/engine_envs/chatterbox/.venv/bin/python'  # set by the optional voice-cloning setup
PREVIEW_DIR = '/content/voice-previews'
workspace = '/content/pdf-audiobook-workspace'
output_dir = '/content/pdf-audiobook-output'

# Your voice library: one WAV per custom voice, kept on Google Drive so it survives runtime resets.
LIBRARY_DIR = '/content/drive/MyDrive/AudiobookFree/voices'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    LIBRARY_DIR = '/content/voice-library'
    print(f'Google Drive is not connected ({exc}); custom voices will only last for this session.')
os.makedirs(LIBRARY_DIR, exist_ok=True)
CAST_ROOT = os.path.join(os.path.dirname(LIBRARY_DIR), 'cast')  # character voices: one folder per book
if '/content/AudiobookFree/src' not in sys.path:
    sys.path.insert(0, '/content/AudiobookFree/src')
cast_tools = importlib.import_module('pdf_audiobook.cast')


def is_custom(voice):
    return voice.endswith('.wav')


def voice_label(voice):
    if is_custom(voice):
        return f"★ {os.path.splitext(os.path.basename(voice))[0]} · your voice"
    return VOICE_LABELS.get(voice, voice)


def library_options():
    clips = sorted(glob.glob(os.path.join(LIBRARY_DIR, '*.wav')), key=str.lower)
    return [(voice_label(path), path) for path in clips]

voice_picker = widgets.Dropdown(
    options=VOICE_OPTIONS,
    value='af_heart',
    description='Narrator voice',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%'),
)
speed_slider = widgets.FloatSlider(
    value=1.0,
    min=0.5,
    max=2.0,
    step=0.05,
    readout_format='.2f',
    description='Speed',
    style={'description_width': 'initial'},
)
sample_text = widgets.Textarea(
    value='The rain had not stopped for three days. "Are you coming with us?" Mara asked, pulling on her coat. Tom glanced at the map, then at the door. "Not tonight," he said quietly.',
    description='Sample text',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%'),
    rows=6,
)
sample_limit = widgets.HTML('<small style="color: gray;">Up to 10,000 characters.</small>')
clone_model_picker = widgets.Dropdown(
    options=[
        ('Turbo: better quality, understands [laugh], [sigh]... tags', 'turbo'),
        ('Original: largest and most expressive, slowest (use Expression, not tags)', 'original'),
        ('Nano: fastest, lowest quality', 'nano'),
    ],
    value='turbo',
    description='Custom voice model',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%'),
)
expression_slider = widgets.FloatSlider(
    value=0.5,
    min=0.25,
    max=2.0,
    step=0.05,
    readout_format='.2f',
    description='Expression',
    style={'description_width': 'initial'},
    layout=widgets.Layout(display='none'),
)


def _on_clone_model_change(change):
    expression_slider.layout.display = '' if change['new'] == 'original' else 'none'


clone_model_picker.observe(_on_clone_model_change, names='value')


def clone_args():
    return ['--clone-model', clone_model_picker.value, '--exaggeration', str(round(expression_slider.value, 2))]

preview_button = widgets.Button(
    description='Preview voice',
    icon='play',
)
compare_picker = widgets.SelectMultiple(
    options=VOICE_OPTIONS,
    rows=8,
    description='Compare with',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%'),
)
compare_hint = widgets.HTML('<small style="color: gray;">Ctrl/Cmd-click to pick several</small>')
voice_hint = widgets.HTML()


def refresh_voice_options(select=None):
    current = select or voice_picker.value
    options = library_options() + VOICE_OPTIONS
    values = [value for _label, value in options]
    voice_picker.options = options
    compare_picker.options = options
    voice_picker.value = current if current in values else 'af_heart'


def _on_voice_change(change):
    custom = is_custom(change['new'] or '')
    speed_slider.disabled = custom
    voice_hint.value = '<small style="color: gray;">Custom voices are narrated by Chatterbox at speed 1.0.</small>' if custom else ''


voice_picker.observe(_on_voice_change, names='value')
refresh_voice_options()
compare_button = widgets.Button(
    description='Compare voices',
)
preview_status = widgets.HTML()
preview_output = widgets.Output()

chapter_mode_picker = widgets.Dropdown(
    options=['original', 'whole', 'custom'],
    value='original',
    description='Chapter mode',
    style={'description_width': 'initial'},
)
chapter_count_box = widgets.BoundedIntText(
    value=4,
    min=2,
    max=50,
    description='Chapter count',
    style={'description_width': 'initial'},
    layout=widgets.Layout(display='none'),
)


def _on_chapter_mode_change(change):
    chapter_count_box.layout.display = '' if change['new'] == 'custom' else 'none'


chapter_mode_picker.observe(_on_chapter_mode_change, names='value')

start_new_box = widgets.Checkbox(
    value=False,
    description="Start new (discard the active conversion's checkpoints)",
    style={'description_width': 'initial'},
)


def render_previews(voices):
    preview_button.disabled = True
    compare_button.disabled = True
    preview_status.value = f"Rendering {len(voices)} voice(s)…"
    preview_output.clear_output()
    t0 = time.time()
    try:
        # Built-in voices run in the Kokoro environment, custom voices in the Chatterbox one.
        builtin = [v for v in voices if not is_custom(v)]
        custom = [v for v in voices if is_custom(v)]
        if custom and not os.path.exists(CB_PY):
            raise RuntimeError('custom voices need the voice-cloning setup cell; run it first')
        previews = []
        failed_output = ''
        for python, flag, group in ((PY, '--voice', builtin), (CB_PY, '--clone', custom)):
            if not group:
                continue
            cmd = [python, '-u', '-m', 'pdf_audiobook.colab', '--preview-dir', PREVIEW_DIR]
            for v in group:
                cmd.extend([flag, v])
            cmd.extend(['--speed', str(round(speed_slider.value, 2)), '--preview-text', sample_text.value])
            if flag == '--clone':
                cmd.extend(clone_args())
            res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            found = []
            for line in res.stdout.splitlines():
                if line.startswith('Preview WAV '):
                    rest = line[len('Preview WAV '):]
                    if ': ' in rest:
                        v_name, path = rest.split(': ', 1)
                        found.append((v_name.strip(), path.strip()))
            if res.returncode != 0 or not found:
                failed_output += res.stdout
            previews.extend(found)
        with preview_output:
            for v_name, path in previews:
                display(HTML(f"<b>{html.escape(voice_label(v_name))}</b>"))
                display(Audio(filename=path))
            if failed_output:
                print(failed_output, end='')
        if failed_output:
            preview_status.value = "Voice preview failed; see output below."
        else:
            elapsed = time.time() - t0
            preview_status.value = f"Rendered {len(previews)} voice(s) in {elapsed:.1f}s."
    except Exception as exc:
        preview_status.value = f"Voice preview failed: {html.escape(str(exc))}"
    finally:
        preview_button.disabled = False
        compare_button.disabled = False


def uploaded_files(upload):
    value = upload.value
    if isinstance(value, dict):  # ipywidgets 7, which Colab uses
        return [(name, bytes(item['content'])) for name, item in value.items()]
    return [(item['name'], bytes(item['content'])) for item in value]


def clear_upload(upload):
    try:
        upload.value = ()
    except Exception:
        upload.value.clear()
        upload._counter = 0


voice_upload = widgets.FileUpload(accept='audio/*,video/*', multiple=True, description='Choose clips')
add_voice_button = widgets.Button(description='Add to library', icon='plus')
add_voice_status = widgets.HTML()
book_upload = widgets.FileUpload(accept='.pdf,.epub', multiple=False, description='Choose book')


def book_pdf():
    """Save the chosen book under /content; return (PDF path, digest of the upload), or None."""
    books = uploaded_files(book_upload)
    if not books:
        return None
    book_name, book_data = books[0]
    pdf_path = '/content/' + os.path.basename(book_name)
    with open(pdf_path, 'wb') as handle:
        handle.write(book_data)
    if pdf_path.lower().endswith('.epub'):
        # The pipeline reads PDFs, so render the EPUB to one. Images are dropped because
        # image-only pages (maps, illustrations) would otherwise be flagged as needing OCR.
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pymupdf', 'pypdf'], check=True)
        import pymupdf, pypdf
        logging.getLogger('pypdf').setLevel(logging.ERROR)
        writer = pypdf.PdfWriter(clone_from=io.BytesIO(pymupdf.open(pdf_path).convert_to_pdf()))
        writer.remove_images()
        pdf_path = pdf_path[:-5] + '.pdf'
        writer.write(pdf_path)
        print('Converted EPUB to', pdf_path)
    return pdf_path, hashlib.sha256(book_data).hexdigest()


VOICE_TMP_DIR = '/content/voice-tmp'
ADD_VOICE_LOG = '/content/add_voice.log'
MAX_VOICE_UPLOAD_BYTES = 20 * 1024 * 1024


def log_add_voice(step):
    # Step log, so a stall shows exactly which step it stopped at.
    try:
        with open(ADD_VOICE_LOG, 'a') as log:
            log.write(f"{time.strftime('%H:%M:%S')} {step}\n")
    except OSError:
        pass


def add_voices(_button):
    add_voice_button.disabled = True
    add_voice_status.value = 'Adding…'
    messages = []
    added = None
    try:
        log_add_voice('callback start')
        clips = uploaded_files(voice_upload)
        log_add_voice(f'upload read: {len(clips)} file(s)')
        if not clips:
            messages.append('Choose one or more clips first.')
        elif sum(len(data) for _name, data in clips) > MAX_VOICE_UPLOAD_BYTES:
            messages.append('Those clips are over 20 MB in total. Trim them to 10–30 seconds (or use MP3) and try again.')
        else:
            os.makedirs(VOICE_TMP_DIR, exist_ok=True)
            for upload_name, data in clips:
                voice_name = re.sub(r'[^\w\- ]+', '', os.path.splitext(os.path.basename(upload_name))[0]).strip() or 'voice'
                raw = os.path.join(VOICE_TMP_DIR, 'upload-' + os.path.basename(upload_name))
                converted = os.path.join(VOICE_TMP_DIR, voice_name + '.wav')
                with open(raw, 'wb') as handle:
                    handle.write(data)
                log_add_voice(f'raw file written: {raw}')
                # Convert locally (never straight into the Drive mount): mono 16-bit 24 kHz WAV, first 30 seconds.
                # -nostdin and stdin=DEVNULL stop ffmpeg from waiting on the kernel's stdin.
                try:
                    res = subprocess.run(
                        ['ffmpeg', '-nostdin', '-y', '-loglevel', 'error', '-i', raw, '-vn', '-ac', '1', '-ar', '24000', '-c:a', 'pcm_s16le', '-t', '30', converted],
                        stdin=subprocess.DEVNULL, capture_output=True, text=True, timeout=120,
                    )
                except subprocess.TimeoutExpired:
                    messages.append(f'Converting {upload_name} took over 2 minutes and was stopped; try a shorter clip.')
                    continue
                finally:
                    os.remove(raw)
                log_add_voice(f'ffmpeg done: exit {res.returncode}')
                if res.returncode != 0:
                    messages.append(f'Could not read {upload_name}: {res.stderr.strip()}')
                    continue
                with wave.open(converted) as clip:
                    seconds = clip.getnframes() / clip.getframerate()
                if seconds <= 5:
                    os.remove(converted)
                    messages.append(f'{upload_name} is only {seconds:.1f} seconds; use at least 6 seconds of clear speech.')
                    continue
                target = os.path.join(LIBRARY_DIR, voice_name + '.wav')
                try:
                    shutil.copy2(converted, target)
                except OSError as exc:
                    messages.append(f'Could not save {voice_name} to {LIBRARY_DIR}: {exc}')
                    continue
                finally:
                    if os.path.exists(converted):
                        os.remove(converted)
                log_add_voice(f'copy to Drive done: {target}')
                messages.append(f'Added ★ {voice_name} ({seconds:.0f} s).')
                added = target
        refresh_voice_options(select=added)
        log_add_voice('refresh done')
    except Exception as exc:
        messages.append(f'Adding the voice failed: {exc}')
        log_add_voice(f'error: {exc!r}')
    finally:
        try:
            clear_upload(voice_upload)
        except Exception:
            pass
        add_voice_button.disabled = False
        add_voice_status.value = '<br>'.join(html.escape(message) for message in messages)


add_voice_button.on_click(add_voices)

# Voice design: Qwen3-TTS VoiceDesign invents a voice from a description and speaks a short sample;
# the sample is saved to the library, and Chatterbox clones it like any other custom voice.
DESIGN_VENV = '/content/voice-design-venv'
DESIGN_PY = DESIGN_VENV + '/bin/python'
voice_design = importlib.import_module('pdf_audiobook.voice_design')


def run_designer(jobs, status):
    """Design (description, path) voices in the voice-design environment, installing it the first time."""
    if not os.path.exists(DESIGN_PY):
        status.value = 'Installing the voice designer (first time only, a few minutes)…'
        for cmd in (['uv', 'venv', '--python', '3.12', DESIGN_VENV], ['uv', 'pip', 'install', '--python', DESIGN_PY, 'qwen-tts']):
            res = subprocess.run(cmd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            if res.returncode != 0:
                shutil.rmtree(DESIGN_VENV, ignore_errors=True)
                raise RuntimeError('installing the voice designer failed: ' + res.stdout.strip()[-1500:])
    batch = '/content/design-jobs.json'
    with open(batch, 'w') as handle:
        json.dump([{'description': description, 'out': out} for description, out in jobs], handle)
    status.value = f'Designing {len(jobs)} voice(s); the first one downloads the model (about 4 GB)…'
    env = {**os.environ, 'PYTHONPATH': '/content/AudiobookFree/src'}
    process = subprocess.Popen([DESIGN_PY, '-u', '-m', 'pdf_audiobook.voice_design', '--batch', batch],
                               stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    output = []
    for line in process.stdout:
        output.append(line)
        if line.startswith('Designed voice '):
            status.value = html.escape(line.split(':', 1)[0]) + '…'
    if process.wait() != 0:
        raise RuntimeError(''.join(output).strip()[-1500:])


design_preset = widgets.Dropdown(
    options=[('Write my own', '')] + [(name, text) for name, text in voice_design.PRESETS.items()],
    value='',
    description='Preset',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%'),
)
design_description = widgets.Textarea(
    placeholder='Describe the voice: gender, age, pitch, texture (gravelly, breathy, smooth), pace, accent, manner…',
    description='Description',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%'),
    rows=3,
)
design_name = widgets.Text(placeholder='e.g. Gruff old man', description='Voice name', style={'description_width': 'initial'})
design_button = widgets.Button(description='Design voice', icon='magic')
design_status = widgets.HTML()
design_output = widgets.Output()


def _on_design_preset(change):
    if change['new']:
        design_description.value = change['new']
        design_name.value = design_preset.label


design_preset.observe(_on_design_preset, names='value')


def design_one(_button):
    design_button.disabled = True
    design_output.clear_output()
    try:
        name = re.sub(r'[^\w\- ]+', '', design_name.value).strip()
        if not name or not design_description.value.strip():
            design_status.value = 'Give the voice a name and a description first.'
            return
        target = os.path.join(LIBRARY_DIR, name + '.wav')
        if os.path.exists(target):
            design_status.value = f'You already have a voice called {html.escape(name)}; pick another name.'
            return
        run_designer([(design_description.value, target)], design_status)
        refresh_voice_options(select=target)
        with design_output:
            display(Audio(filename=target))
        design_status.value = f'Added ★ {html.escape(name)}. Design again for a different take; each one comes out a little different.'
    except Exception as exc:
        design_status.value = f'Designing the voice failed: {html.escape(str(exc))}'
    finally:
        design_button.disabled = False


design_button.on_click(design_one)

# Character voices: Gemini labels who says each quote; each character gets their own voice.
narration_picker = widgets.Dropdown(
    options=[('One voice', 'single'), ('Character voices (Gemini): each character gets their own voice', 'cast')],
    value='single',
    description='Narration',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%'),
)
gemini_key = widgets.Password(
    description='Gemini API key (optional)',
    placeholder="Leave blank to use Colab's built-in Gemini",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%'),
)
make_script_button = widgets.Button(description='Make script', icon='magic')
design_cast_box = widgets.Checkbox(
    value=True,
    description='Design a unique voice for each character from the book (slower; uncheck to use existing voices)',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='100%'),
)
own_voice_from = widgets.BoundedIntText(
    value=5,
    min=1,
    max=100000,
    description='Own voice from (lines)',
    style={'description_width': 'initial'},
)
cast_status = widgets.HTML()
cast_voices = widgets.VBox()
cast_state = {}


def cast_work_dir(pdf_path, digest):
    mode = chapter_mode_picker.value
    count = chapter_count_box.value if mode == 'custom' else ''
    stem = re.sub(r'[^\w\- ]+', '', os.path.splitext(os.path.basename(pdf_path))[0]).strip()[:40] or 'book'
    return os.path.join(CAST_ROOT, f'{stem}-{digest[:12]}-{mode}{count}')


def designed_voice_path(work, name):
    safe = re.sub(r'[^\w\- ]+', '', name).strip() or 'character'
    return os.path.join(work, 'voices', safe + '.wav')


def show_cast(work):
    # Characters with fewer lines than "Own voice from" are read by the narrator and not listed.
    people = cast_tools.cast_list(work)
    saved_path = os.path.join(work, 'settings.json')
    saved = {}
    if os.path.exists(saved_path):
        with open(saved_path) as handle:
            saved = json.load(handle).get('voices', {})
    saved.update({name: picker.value for name, picker in cast_state.get('pickers', {}).items()})
    major = [person for person in people if person['lines'] >= own_voice_from.value]
    suggested = cast_tools.suggest_voices(major, voice_picker.value, cast_tools.KOKORO_BEST_FIRST)
    suggested.update(cast_state.get('recommended', {}))
    suggested.update(cast_state.get('designed', {}))
    designed_options = [(f'✦ {name} · designed for this book', path) for name, path in cast_state.get('designed', {}).items()]
    options = [('Narrator voice', 'narrator')] + designed_options + library_options() + VOICE_OPTIONS
    values = {value for _label, value in options}
    pickers = {}
    rows = [widgets.HTML(f'<small style="color: gray;">{len(major)} character(s) with their own voice. The narrator uses the Narrator voice above.</small>')]
    for person in major:
        name, count = person['name'], person['lines']
        lines = f"{count} line{'s' if count != 1 else ''}"
        about = f"{name} · {person['gender']} · {lines}" if person['gender'] != 'unknown' else f'{name} · {lines}'
        pickers[name] = widgets.Dropdown(
            options=options,
            value=saved[name] if saved.get(name) in values else suggested.get(name, 'narrator'),
            description=about,
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='100%'),
        )
        rows.append(pickers[name])
        notes = [note for note in (person['about'], person['voice'] and f"Sounds: {person['voice']}") if note]
        if notes:
            rows.append(widgets.HTML('<small style="color: gray; margin-left: 1em;">' + html.escape(' — '.join(notes)) + '</small>'))
    if len(people) > len(major):
        minor = len(people) - len(major)
        rows.append(widgets.HTML(f'<small style="color: gray;">{minor} minor character(s) with fewer than {own_voice_from.value} lines use the narrator voice. '
                                 'Lower "Own voice from" to give them their own.</small>'))
    cast_voices.children = rows
    cast_state.update(work=work, pickers=pickers)


def save_cast_settings():
    work = cast_state['work']
    voices = {'narrator': voice_picker.value, **{name: picker.value for name, picker in cast_state['pickers'].items()}}
    settings = {
        'voices': voices,
        'speed': round(speed_slider.value, 2),
        'clone_model': clone_model_picker.value,
        'exaggeration': round(expression_slider.value, 2),
    }
    settings_path = os.path.join(work, 'settings.json')
    with open(settings_path, 'w') as handle:
        json.dump(settings, handle, indent=1)
    return work, settings_path, voices


def make_script(_button):
    make_script_button.disabled = True
    cast_voices.children = ()
    cast_preview_output.clear_output()
    cast_state.clear()
    try:
        book = book_pdf()
        if book is None:
            cast_status.value = 'Choose your book under "Book" first.'
            return
        pdf_path, digest = book
        work = cast_work_dir(pdf_path, digest)
        cast_status.value = 'Reading the book…'
        cmd = [PY, '-u', '-m', 'pdf_audiobook.cast', 'extract', pdf_path, work, '--chapter-mode', chapter_mode_picker.value]
        if chapter_mode_picker.value == 'custom':
            cmd += ['--chapter-count', str(chapter_count_box.value)]
        res = subprocess.run(cmd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        if res.returncode != 0:
            raise RuntimeError(res.stdout.strip()[-1500:])
        script = os.path.join(work, 'script.txt')
        key = gemini_key.value.strip()
        ask = cast_tools.gemini_api(key) if key else cast_tools.gemini_colab()
        if os.path.exists(script):
            note = 'Loaded your saved script.'
        else:
            cast_tools.make_script(work, ask, progress=lambda line: setattr(cast_status, 'value', html.escape(f'Gemini is reading the book… {line}')))
            note = 'Script ready.'
        # Each main character gets a voice designed from Gemini's description of them; they are saved
        # with the book's script, so only characters without one are designed on later clicks.
        designed = {}
        if design_cast_box.value:
            main = [person for person in cast_tools.cast_list(work) if person['lines'] >= own_voice_from.value and person['voice']][:30]
            jobs = {person['name']: (person['voice'], designed_voice_path(work, person['name'])) for person in main}
            missing = [(description, path) for description, path in jobs.values() if not os.path.exists(path)]
            try:
                if missing:
                    run_designer(missing, cast_status)
                designed = {name: path for name, (_description, path) in jobs.items() if os.path.exists(path)}
            except Exception as exc:
                note += f' (Designing voices failed: {exc}; using existing voices.)'
        cast_state['designed'] = designed
        # Gemini recommends a voice per character as the fallback; answers are cached.
        cast_status.value = 'Gemini is casting the voices…'
        people = [person for person in cast_tools.cast_list(work) if person['lines'] >= 2][:40]
        try:
            cast_state['recommended'] = cast_tools.recommend_voices(work, ask, people, cast_tools.voice_choices(library_options(), voice_picker.value))
        except Exception as exc:
            note += f' (Gemini could not recommend voices: {exc}; using defaults.)'
        show_cast(work)
        cast_status.value = (f'{note} Pick a voice for each character, then run the last cell.<br>'
                             f'<small style="color: gray;">Script: {html.escape(script)}. Edit it to fix who says a line, then click Make script again; '
                             'delete its folder to have Gemini start over.</small>')
    except Exception as exc:
        cast_status.value = f'Making the script failed: {html.escape(str(exc))}'
    finally:
        make_script_button.disabled = False


make_script_button.on_click(make_script)


def preview_cast(_button):
    # Each listed character (and the narrator) reads one of their own lines in the chosen voice.
    # The clips are the same cached lines generation uses, so previewing costs no extra time later.
    if not cast_state:
        cast_status.value = 'Click "Make script" first.'
        return
    preview_cast_button.disabled = True
    cast_preview_output.clear_output()
    try:
        work, settings_path, voices = save_cast_settings()
        cast_tools.write_preview(work, ['narrator'] + list(cast_state['pickers']))
        uses_custom = any(is_custom(v) for v in voices.values())
        if uses_custom and not os.path.exists(CB_PY):
            raise RuntimeError('a character uses a ★ custom voice; run the voice-cloning setup cell first')
        cast_status.value = 'Rendering the cast preview…'
        for python, engine in ((PY, 'kokoro'), (CB_PY, 'clone')):
            if engine == 'clone' and not uses_custom:
                continue
            res = subprocess.run([python, '-u', '-m', 'pdf_audiobook.cast', 'render', work, '--settings', settings_path, '--engine', engine, '--script', 'preview.txt'],
                                 stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            if res.returncode != 0:
                raise RuntimeError(res.stdout.strip()[-1500:])
        with cast_preview_output:
            for speaker, path in cast_tools.preview_lines(work, settings_path):
                voice = voices.get(speaker) if voices.get(speaker, 'narrator') != 'narrator' else voice_picker.value
                display(HTML(f'<b>{html.escape(speaker)}</b> <small style="color: gray;">({html.escape(voice_label(voice))})</small>'))
                display(Audio(filename=str(path)))
        cast_status.value = 'Cast preview ready below. Change any voice and preview again, then run the last cell.'
    except Exception as exc:
        cast_status.value = f'Cast preview failed: {html.escape(str(exc))}'
    finally:
        preview_cast_button.disabled = False


preview_cast_button = widgets.Button(description='Preview cast', icon='play')
cast_preview_output = widgets.Output()
preview_cast_button.on_click(preview_cast)
own_voice_from.observe(lambda _change: show_cast(cast_state['work']) if cast_state else None, names='value')
cast_section = widgets.VBox([
    widgets.HTML('<h3>Character voices</h3><small style="color: gray;">Gemini reads the book and labels who says each quote; '
                 "the book's words never change. Set the chapter mode above first. Uses Colab's built-in Gemini (your Google login); "
                 'if it is unavailable or busy, paste a free key from aistudio.google.com/apikey.</small>'),
    gemini_key,
    design_cast_box,
    make_script_button,
    cast_status,
    own_voice_from,
    cast_voices,
    preview_cast_button,
    cast_preview_output,
], layout=widgets.Layout(display='none'))


def _on_narration_change(change):
    cast_section.layout.display = '' if change['new'] == 'cast' else 'none'


narration_picker.observe(_on_narration_change, names='value')
preview_button.on_click(lambda _b: render_previews([voice_picker.value]))
compare_button.on_click(
    lambda _b: render_previews(
        [voice_picker.value] + [v for v in compare_picker.value if v != voice_picker.value]
    )
)

panel = widgets.VBox([
    narration_picker,
    widgets.HTML('<h3>Voice</h3>'),
    voice_picker,
    voice_hint,
    clone_model_picker,
    expression_slider,
    speed_slider,
    sample_text,
    sample_limit,
    preview_button,
    widgets.VBox([compare_picker, compare_hint]),
    compare_button,
    preview_status,
    preview_output,
    widgets.HTML('<h3>Add a voice</h3><small style="color: gray;">10–30 seconds of one person speaking clearly, no music. '
                 'The file name becomes the voice name; it is saved to your Google Drive under AudiobookFree/voices.</small>'),
    widgets.HBox([voice_upload, add_voice_button]),
    add_voice_status,
    widgets.HTML('<h3>Design a voice</h3><small style="color: gray;">Describe any voice, or pick a preset, and a new one is made '
                 'for you (Qwen3-TTS VoiceDesign) and saved to your library.</small>'),
    design_preset,
    design_description,
    widgets.HBox([design_name, design_button]),
    design_status,
    design_output,
    widgets.HTML('<h3>Book</h3>'),
    book_upload,
    widgets.HTML('<small style="color: gray;">PDF with selectable text, or EPUB. Then run the last cell to generate.</small>'),
    widgets.HTML('<h3>Conversion</h3>'),
    chapter_mode_picker,
    chapter_count_box,
    start_new_box,
    cast_section,
])
display(panel)

In [ ]:
# One long, resumable headless run. Rerun this cell after a disconnect with the same book/settings.
from google.colab import files


def stream(cmd):
    """Run one step, echoing its output; return the verified M4B path if it printed one."""
    process = subprocess.Popen(cmd, stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    verified = None
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        if line.startswith('Verified M4B: '):
            verified = line.removeprefix('Verified M4B: ').strip()
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'Colab conversion failed with exit code {return_code}')
    return verified


def download(verified_path):
    if not verified_path:
        raise RuntimeError('The runner did not report a verified M4B path.')
    files.download(verified_path)


if narration_picker.value == 'cast':
    # Character voices: every line is cached on Drive, so a rerun after a disconnect only renders what is missing.
    if not cast_state:
        print('Character voices are on: click "Make script" under "Character voices" in the panel first, then run this cell.')
    else:
        work, settings_path, voices = save_cast_settings()
        uses_custom = any(is_custom(v) for v in voices.values())
        if uses_custom and not os.path.exists(CB_PY):
            raise RuntimeError('A character uses a ★ custom voice; run the voice-cloning setup cell first.')
        print(f'Character voices: {len(voices) - 1} character(s), narrator={voice_picker.value}, script={work}')
        cast_cmd = ['-u', '-m', 'pdf_audiobook.cast']
        stream([PY, *cast_cmd, 'render', work, '--settings', settings_path, '--engine', 'kokoro'])
        if uses_custom:
            stream([CB_PY, *cast_cmd, 'render', work, '--settings', settings_path, '--engine', 'clone'])
        download(stream([PY, *cast_cmd, 'build', work, '--settings', settings_path, '--output-dir', output_dir]))
else:
    book = book_pdf()
    if book is None:
        print('Choose your book (PDF or EPUB) under "Book" in the panel above, then run this cell.')
    else:
        pdf_path, _digest = book
        voice = voice_picker.value
        custom = is_custom(voice)
        speed = 1.0 if custom else round(speed_slider.value, 2)  # the slider can report 1.1500000000000001
        chapter_mode = chapter_mode_picker.value
        count = chapter_count_box.value if chapter_mode == 'custom' else None
        start_new = start_new_box.value
        print(f'voice={voice} speed={speed} chapter_mode={chapter_mode} count={count} start_new={start_new} workspace={workspace} output_dir={output_dir}')
        # Custom voices run in the Chatterbox environment from the voice-cloning setup cell.
        python, voice_flag = (CB_PY, '--clone') if custom else (PY, '--voice')
        cmd = [python, '-u', '-m', 'pdf_audiobook.colab', pdf_path, '--workspace-root', workspace, '--output-dir', output_dir, voice_flag, voice, '--speed', str(speed), '--chapter-mode', chapter_mode]
        if count is not None:
            cmd += ['--chapter-count', str(count)]
        if custom:
            cmd += clone_args()
        if start_new:
            cmd += ['--start-new']
        download(stream(cmd))
